# TraceProfiler: Step 1 - Full MAS Repository Understanding & Ingestion
### Multi-Agent System AST Analyzer for all 10 Workflows (78 Functional Agents)

This notebook performs **static, non-executing AST analysis** of all 10 multi-agent workflows in this repository.

### Target Workflows Analyzed:
1. **Invoice Processing & Accounts Payable** (8 agents)
2. **Customer Support Ticket Resolution** (6 agents)
3. **Procurement & Supplier Selection** (9 agents)
4. **Sales Lead Qualification** (7 agents)
5. **Recruitment & Resume Screening** (8 agents)
6. **Inventory Replenishment Planning** (6 agents)
7. **Contract Review & Compliance** (10 agents)
8. **IT Incident Triage & Remediation** (7 agents)
9. **Employee Expense Reimbursement** (8 agents)
10. **Market & Competitor Research** (9 agents)

**Safety Principle:** This analysis reads source files as text and parses the AST. It never imports or executes modules, preventing unintended API calls, LLM spending, or runtime side effects.

## 1. Environment & Target Repository Setup

Auto-detects whether running locally in the repository root or inside Google Colab.

In [ ]:
import os, sys, json, zipfile, shutil, textwrap, re, ast, hashlib, csv
from pathlib import Path
from collections import defaultdict, Counter

# Auto-detect repository root (local workspace vs Colab)
local_root = Path.cwd()
if (local_root / "Agents").exists():
    ROOT = local_root
    print(f"Running in local repository at: {ROOT}")
else:
    ROOT = Path("/content/target_mas")
    ROOT.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import files
        print("Please upload a ZIP of the repository or individual files:")
        uploaded = files.upload()
        zip_files = [name for name in uploaded if name.lower().endswith(".zip")]
        if zip_files:
            with zipfile.ZipFile(zip_files[0]) as zf:
                zf.extractall(ROOT)
            print(f"Extracted {zip_files[0]} into {ROOT}")
    except ImportError:
        print(f"Using directory: {ROOT}")

# If extracted into a single subdirectory, unwrap it
children = list(ROOT.iterdir())
if len(children) == 1 and children[0].is_dir() and not (ROOT / "Agents").exists():
    ROOT = children[0]

print(f"Repository root: {ROOT}")
print(f"Python files   : {len(list(ROOT.rglob('*.py')))}")
print(f"JSON files     : {len(list(ROOT.rglob('*.json')))}")

## 2. Run AST Workflow Analyzer Engine

Executes full static parsing across all Python modules, normalizes workflow manifests, compares declared vs. observed execution orders, and flags architectural discrepancies.

In [ ]:
from tools.ast_workflow_analyzer import ASTWorkflowAnalyzer, print_cli_summary

analyzer = ASTWorkflowAnalyzer(root_dir=ROOT)
report = analyzer.run_full_analysis()
print_cli_summary(report)

## 3. Export Reports & Download Artifacts

Generates consolidated `target_mas_ingestion_report.json` and CSV summaries in `reports/traceprofiler`.

In [ ]:
out_dir = analyzer.export_reports()
print(f"Reports exported to: {out_dir}")
print("\n--- Consolidated Repository Reports ---")
for p in sorted(out_dir.iterdir()):
    if p.is_file():
        print(f"  - {p.name} ({p.stat().st_size:,} bytes)")

print("\n--- Separated Workflow Subdirectories ---")
for d in sorted(out_dir.iterdir()):
    if d.is_dir():
        files_count = len(list(d.iterdir()))
        print(f"  [Folder] {d.name}/ ({files_count} reports)")

try:
    from google.colab import files
    for p in out_dir.rglob("*"):
        if p.is_file():
            files.download(str(p))
except ImportError:
    print("\nLocal environment: files ready in reports/traceprofiler/")